In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS AIRLINE_DB;
CREATE SCHEMA IF NOT EXISTS AIRLINE_DB.RAW;
CREATE SCHEMA IF NOT EXISTS AIRLINE_DB.STAGE;
CREATE SCHEMA IF NOT EXISTS AIRLINE_DB.DWH;
CREATE SCHEMA IF NOT EXISTS AIRLINE_DB.AUDIT;

USE DATABASE AIRLINE_DB;

CREATE
OR REPLACE STAGE AIRLINE_DB.RAW.MY_STAGE;

-- FILE FORMAT
CREATE
OR REPLACE FILE FORMAT AIRLINE_DB.RAW.AIRLINE_CSV 
TYPE = CSV SKIP_HEADER = 1
FIELD_OPTIONALLY_ENCLOSED_BY = '"' 
NULL_IF = ('', 'NULL', 'null', 'N/A', ' ', '-')
DATE_FORMAT = 'MM/DD/YYYY';

-- LOG Table
CREATE
OR REPLACE TABLE AIRLINE_DB.AUDIT.LOG (
    LOG_ID NUMBER AUTOINCREMENT START 1 INCREMENT 1,
    PIPELINE_STEP STRING,
    TARGET_TABLE STRING,
    ROWS_INSERTED NUMBER,
    ROWS_UPDATED NUMBER,
    EXECUTION_TIME TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    STATUS STRING,
    ERROR_MESSAGE STRING
);

-- RAW: AIRLINE_RAW Table
CREATE
OR REPLACE TABLE AIRLINE_DB.RAW.AIRLINE_RAW(
    "Column1" INT,
    "Passenger ID" VARCHAR(10),  
    "First Name" VARCHAR,
    "Last Name" VARCHAR,
    "Gender" VARCHAR,
    "Age" INT,
    "Nationality" VARCHAR,
    "Airport Name" VARCHAR,
    "Airport Country Code" VARCHAR(2),
    "Country Name" VARCHAR,
    "Airport Continent" VARCHAR(3),
    "Continents" VARCHAR,
    "Departure Date" DATE,     
    "Arrival Airport" VARCHAR(3),
    "Pilot Name" VARCHAR,
    "Flight Status" VARCHAR,
    "Ticket Type" VARCHAR,
    "Passenger Status" VARCHAR
);

-- STREAM: AIRLINE_STREAM
CREATE OR REPLACE STREAM AIRLINE_DB.RAW.AIRLINE_STREAM 
ON TABLE AIRLINE_DB.RAW.AIRLINE_RAW
APPEND_ONLY = TRUE;

-- STAGE: AIRLINE_STAGE
CREATE TABLE IF NOT EXISTS AIRLINE_DB.STAGE.AIRLINE_STAGE (  
    ID INT,
    PASSENGER_ID VARCHAR(10),  
    FIRST_NAME STRING,
    LAST_NAME STRING,
    GENDER STRING,
    AGE INT,
    NATIONALITY STRING,
    AIRPORT_NAME STRING,
    AIRPORT_COUNTRY_CODE VARCHAR(2),
    COUNTRY_NAME STRING,
    AIRPORT_CONTINENT VARCHAR(3),
    CONTINENTS STRING,
    DEPARTURE_DATE DATE,     
    ARRIVAL_AIRPORT VARCHAR(3),
    PILOT_NAME STRING,
    FLIGHT_STATUS STRING,
    TICKET_TYPE STRING,
    PASSENGER_STATUS STRING
);

In [ ]:
%%sql
CREATE OR REPLACE PROCEDURE AIRLINE_DB.RAW.LOAD_RAW_AND_WRITE_LOG()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
    V_ROWS_LOADED INT DEFAULT 0;
    V_ERROR_MSG STRING DEFAULT '';
BEGIN
    -- 1. Execute COPY
    COPY INTO AIRLINE_DB.RAW.AIRLINE_RAW
    FROM @AIRLINE_DB.RAW.MY_STAGE
    FILE_FORMAT = (FORMAT_NAME = AIRLINE_DB.RAW.AIRLINE_CSV)
    ON_ERROR = SKIP_FILE
    PURGE = TRUE
    TRUNCATECOLUMNS = TRUE;


    -- 2. Automatically get total loaded rows using built-in variable
    V_ROWS_LOADED := SQLROWCOUNT;

    -- 3. Log success
    INSERT INTO AIRLINE_DB.AUDIT.LOG (
        PIPELINE_STEP,
        TARGET_TABLE,
        ROWS_INSERTED,
        ROWS_UPDATED,
        STATUS
    )
    VALUES (
        'LOAD_STAGE_TO_RAW',
        'AIRLINE_DB.RAW.AIRLINE_RAW',
        :V_ROWS_LOADED,
        0,
        'SUCCESS'
    );

    RETURN 'Success: Loaded ' || :V_ROWS_LOADED || ' rows into RAW layer.';

EXCEPTION
    WHEN OTHER THEN

        V_ERROR_MSG := SQLERRM;

        INSERT INTO AIRLINE_DB.AUDIT.LOG (
            PIPELINE_STEP,
            TARGET_TABLE,
            ROWS_INSERTED,
            ROWS_UPDATED,
            STATUS,
            ERROR_MESSAGE
        )
        VALUES (
            'LOAD_STAGE_TO_RAW',
            'AIRLINE_DB.RAW.AIRLINE_RAW',
            0,
            0,
            'FAILED',
            :V_ERROR_MSG
        );

        RETURN 'Failed to load data. Error: ' || :V_ERROR_MSG;

END;
$$;

In [ ]:
%%sql
CREATE OR REPLACE PROCEDURE AIRLINE_DB.STAGE.TRANSFORM_RAW_TO_STAGE()
RETURNS VARCHAR
LANGUAGE SQL
AS 
$$
BEGIN 
    TRUNCATE TABLE AIRLINE_DB.STAGE.AIRLINE_STAGE;

    INSERT INTO AIRLINE_DB.STAGE.AIRLINE_STAGE 
    SELECT  
        "Column1" AS ID,
        "Passenger ID" AS PASSENGER_ID,  
        "First Name" AS FIRST_NAME,
        "Last Name" AS LAST_NAME,
        UPPER(TRIM("Gender")) AS GENDER,                  -- Gender: normalize
        TRY_TO_NUMBER("Age") AS AGE,                      -- Age: convert to number safely
        UPPER(TRIM("Nationality")) AS NATIONALITY,
        TRIM("Airport Name") AS AIRPORT_NAME,
        UPPER(TRIM("Airport Country Code")) AS AIRPORT_COUNTRY_CODE,
        UPPER(TRIM("Country Name")) AS COUNTRY_NAME,
        UPPER(TRIM("Airport Continent")) AS AIRPORT_CONTINENT,
        UPPER(TRIM("Continents")) AS CONTINENTS,
        TRY_TO_DATE("Departure Date") AS DEPARTURE_DATE,  -- Departure Date: safe date parsing   
        UPPER(TRIM("Arrival Airport")) AS ARRIVAL_AIRPORT,
        TRIM("Pilot Name") AS PILOT_NAME,
        UPPER(TRIM("Flight Status")) AS FLIGHT_STATUS,
        UPPER(TRIM("Ticket Type")) AS TICKET_TYPE,
        UPPER(TRIM("Passenger Status")) AS PASSENGER_STATUS
    FROM AIRLINE_DB.RAW.AIRLINE_STREAM;
    
RETURN 'SUCCESS: Data successfully transformed and loaded into Stage.';

EXCEPTION 
    WHEN OTHER THEN
        RETURN 'FAILED: Error Code ' || SQLCODE || ' - ' || SQLERRM;

END;
$$;

In [ ]:
%%sql
-- DIM: Passengers
CREATE OR REPLACE TABLE AIRLINE_DB.DWH.DIM_PASSENGER (
    PASSENGER_SK NUMBER AUTOINCREMENT PRIMARY KEY, 
    PASSENGER_ID VARCHAR(10),
    FIRST_NAME STRING,
    LAST_NAME STRING,
    GENDER STRING,
    AGE NUMBER,
    NATIONALITY STRING
);

-- DIM: Airports
CREATE OR REPLACE TABLE AIRLINE_DB.DWH.DIM_AIRPORT (
    AIRPORT_ID NUMBER PRIMARY KEY AUTOINCREMENT,
    AIRPORT_NAME STRING ,
    AIRPORT_COUNTRY_CODE VARCHAR(2),
    COUNTRY_NAME STRING,
    AIRPORT_CONTINENT VARCHAR(3),
    CONTINENTS STRING
);

-- DIM: Dates
CREATE OR REPLACE TABLE AIRLINE_DB.DWH.DIM_DATE (
    CALENDAR_DATE DATE PRIMARY KEY, 
    FLIGHT_YEAR NUMBER,
    FLIGHT_MONTH NUMBER,
    FLIGHT_DAY NUMBER
);

-- DIM: Pilots
CREATE OR REPLACE TABLE AIRLINE_DB.DWH.DIM_PILOT (
    PILOT_ID NUMBER AUTOINCREMENT PRIMARY KEY,
    PILOT_NAME STRING
);

-- DIM: Statuses and Categories
CREATE OR REPLACE TABLE AIRLINE_DB.DWH.DIM_FLIGHT_DETAILS (
    DETAILS_KEY NUMBER AUTOINCREMENT PRIMARY KEY,
    FLIGHT_STATUS STRING, -- e.g., 'On Time', 'Delayed', 'Cancelled'
    TICKET_TYPE STRING,   -- e.g., 'Economy', 'Business', 'First Class'
    PASSENGER_STATUS STRING -- e.g., 'Checked-in', 'Boarded', 'No-show'
);


-- FACT: Flights
CREATE OR REPLACE TABLE AIRLINE_DB.DWH.FACT_FLIGHTS (
    FlIGHT_ID NUMBER AUTOINCREMENT PRIMARY KEY, 
    ID NUMBER,                      -- Column 1 from RAW
    PASSENGER_SK NUMBER,            -- FK   
    AIRPORT_ID NUMBER,              -- FK
    DEPARTURE_DATE DATE,            -- FK
    ARRIVAL_AIRPORT_CODE VARCHAR(3),
    PILOT_ID NUMBER,                -- FK
    DETAILS_KEY NUMBER,             -- FK
    
    -- Defining the Foreign Keys
    CONSTRAINT fk_passenger FOREIGN KEY (PASSENGER_SK) REFERENCES AIRLINE_DB.DWH.DIM_PASSENGER(PASSENGER_SK),
    CONSTRAINT fk_airport FOREIGN KEY (AIRPORT_ID) REFERENCES AIRLINE_DB.DWH.DIM_AIRPORT(AIRPORT_ID),
    CONSTRAINT fk_date FOREIGN KEY (DEPARTURE_DATE) REFERENCES AIRLINE_DB.DWH.DIM_DATE(CALENDAR_DATE),
    CONSTRAINT fk_pilot FOREIGN KEY (PILOT_ID) REFERENCES AIRLINE_DB.DWH.DIM_PILOT(PILOT_ID),
    CONSTRAINT fk_details FOREIGN KEY (DETAILS_KEY) REFERENCES AIRLINE_DB.DWH.DIM_FLIGHT_DETAILS(DETAILS_KEY)
);

In [ ]:
%%sql
CREATE OR REPLACE PROCEDURE AIRLINE_DB.DWH.LOAD_STAGE_TO_DWH()
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN

    -- 1. LOAD DIMENSIONS (Against dirty data)

    -- DIM_PASSENGER
    MERGE INTO AIRLINE_DB.DWH.DIM_PASSENGER T
    USING (
        SELECT PASSENGER_ID, FIRST_NAME, LAST_NAME, GENDER, AGE, NATIONALITY 
        FROM AIRLINE_DB.STAGE.AIRLINE_STAGE
        WHERE PASSENGER_ID IS NOT NULL
        -- Forces exactly 1 row per Passenger ID
        QUALIFY ROW_NUMBER() OVER (PARTITION BY PASSENGER_ID ORDER BY AGE DESC) = 1
    ) S
    ON T.PASSENGER_ID = S.PASSENGER_ID
    WHEN MATCHED THEN 
        UPDATE SET 
            T.FIRST_NAME = S.FIRST_NAME,
            T.LAST_NAME = S.LAST_NAME,
            T.GENDER = S.GENDER,
            T.AGE = S.AGE,
            T.NATIONALITY = S.NATIONALITY
    WHEN NOT MATCHED THEN 
        INSERT (PASSENGER_ID, FIRST_NAME, LAST_NAME, GENDER, AGE, NATIONALITY)
        VALUES (S.PASSENGER_ID, S.FIRST_NAME, S.LAST_NAME, S.GENDER, S.AGE, S.NATIONALITY);

    -- DIM_AIRPORT
    MERGE INTO AIRLINE_DB.DWH.DIM_AIRPORT T
    USING (
        SELECT AIRPORT_NAME, AIRPORT_COUNTRY_CODE, COUNTRY_NAME, AIRPORT_CONTINENT, CONTINENTS
        FROM AIRLINE_DB.STAGE.AIRLINE_STAGE
        WHERE AIRPORT_NAME IS NOT NULL
        -- BULLETPROOF: Forces exactly 1 row per Airport Name
        QUALIFY ROW_NUMBER() OVER (PARTITION BY AIRPORT_NAME ORDER BY COUNTRY_NAME) = 1 
    ) S
    ON T.AIRPORT_NAME = S.AIRPORT_NAME
    WHEN NOT MATCHED THEN
        INSERT (AIRPORT_NAME, AIRPORT_COUNTRY_CODE, COUNTRY_NAME, AIRPORT_CONTINENT, CONTINENTS)
        VALUES (S.AIRPORT_NAME, S.AIRPORT_COUNTRY_CODE, S.COUNTRY_NAME, S.AIRPORT_CONTINENT, S.CONTINENTS);

    -- DIM_DATE 
    MERGE INTO AIRLINE_DB.DWH.DIM_DATE T
    USING (
        SELECT DISTINCT 
               DEPARTURE_DATE AS CALENDAR_DATE,
               YEAR(DEPARTURE_DATE) AS FLIGHT_YEAR,
               MONTH(DEPARTURE_DATE) AS FLIGHT_MONTH,
               DAY(DEPARTURE_DATE) AS FLIGHT_DAY
        FROM AIRLINE_DB.STAGE.AIRLINE_STAGE
        WHERE DEPARTURE_DATE IS NOT NULL
    ) S
    ON T.CALENDAR_DATE = S.CALENDAR_DATE
    WHEN NOT MATCHED THEN
        INSERT (CALENDAR_DATE, FLIGHT_YEAR, FLIGHT_MONTH, FLIGHT_DAY)
        VALUES (S.CALENDAR_DATE, S.FLIGHT_YEAR, S.FLIGHT_MONTH, S.FLIGHT_DAY);

    -- DIM_PILOT
    MERGE INTO AIRLINE_DB.DWH.DIM_PILOT T
    USING (
        SELECT PILOT_NAME 
        FROM AIRLINE_DB.STAGE.AIRLINE_STAGE
        WHERE PILOT_NAME IS NOT NULL
        -- BULLETPROOF: Forces exactly 1 row per Pilot
        QUALIFY ROW_NUMBER() OVER (PARTITION BY PILOT_NAME ORDER BY PILOT_NAME) = 1
    ) S
    ON T.PILOT_NAME = S.PILOT_NAME
    WHEN NOT MATCHED THEN
        INSERT (PILOT_NAME) VALUES (S.PILOT_NAME);

    -- DIM_FLIGHT_DETAILS (DISTINCT is safe here because the PK *is* the combination of these 3 columns)
    MERGE INTO AIRLINE_DB.DWH.DIM_FLIGHT_DETAILS T
    USING (
        SELECT DISTINCT FLIGHT_STATUS, TICKET_TYPE, PASSENGER_STATUS
        FROM AIRLINE_DB.STAGE.AIRLINE_STAGE
    ) S
    ON  T.FLIGHT_STATUS = S.FLIGHT_STATUS 
    AND T.TICKET_TYPE = S.TICKET_TYPE 
    AND T.PASSENGER_STATUS = S.PASSENGER_STATUS
    WHEN NOT MATCHED THEN
        INSERT (FLIGHT_STATUS, TICKET_TYPE, PASSENGER_STATUS)
        VALUES (S.FLIGHT_STATUS, S.TICKET_TYPE, S.PASSENGER_STATUS);


    -- 2. LOAD FACT TABLE 
    INSERT INTO AIRLINE_DB.DWH.FACT_FLIGHTS (
        ID, PASSENGER_SK, AIRPORT_ID, DEPARTURE_DATE, ARRIVAL_AIRPORT_CODE, PILOT_ID, DETAILS_KEY
    )
    SELECT 
        S.ID,
        DP.PASSENGER_SK,
        DA.AIRPORT_ID,
        S.DEPARTURE_DATE,
        S.ARRIVAL_AIRPORT, 
        PI.PILOT_ID,
        FD.DETAILS_KEY
    FROM AIRLINE_DB.STAGE.AIRLINE_STAGE S
    LEFT JOIN AIRLINE_DB.DWH.DIM_PASSENGER DP ON S.PASSENGER_ID = DP.PASSENGER_ID
    LEFT JOIN AIRLINE_DB.DWH.DIM_AIRPORT DA ON S.AIRPORT_NAME = DA.AIRPORT_NAME
    LEFT JOIN AIRLINE_DB.DWH.DIM_PILOT PI ON S.PILOT_NAME = PI.PILOT_NAME
    LEFT JOIN AIRLINE_DB.DWH.DIM_FLIGHT_DETAILS FD 
           ON S.FLIGHT_STATUS = FD.FLIGHT_STATUS 
          AND S.TICKET_TYPE = FD.TICKET_TYPE 
          AND S.PASSENGER_STATUS = FD.PASSENGER_STATUS;

    RETURN 'Success: Stage data successfully loaded into Star Schema.';
END;
$$;